In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Solve the logistic regression problem on a GPU. Given a feature matrix $X$ of size $n\_samples \times n\_features$ and a binary target vector $y$ of size $n\_samples$ (containing only 0s and 1s), compute the coefficient vector $\beta$ that maximizes the log-likelihood:
  $$ \max_{\beta} \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1-y_i) \log(1-p_i) \right] $$

  where $p_i = \sigma(X_i^T \beta)$ and $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the sigmoid function.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final coefficients must be stored in the <code>beta</code> vector</li>
  <li>The target vector <code>y</code> contains only binary values (0 and 1)</li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
$X$ (samples × features):
$$
\begin{bmatrix}
2.0 & 1.0 \\
1.0 & 2.0 \\
3.0 & 3.0 \\
1.5 & 2.5 \\
-1.0 & -2.0 \\
-2.0 & -1.0 \\
-1.5 & -2.5 \\
-3.0 & -3.0
\end{bmatrix}
$$
$y$:
$$
\begin{bmatrix}
1 \\
1 \\
1 \\
0 \\
0 \\
0 \\
1 \\
0
\end{bmatrix}
$$
Output:<br>
$\beta$:
$$
\begin{bmatrix}
2.26 \\
-1.29
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>n_samples</code> ≤ 100,000</li>
  <li>1 ≤ <code>n_features</code> ≤ 1,000</li>
  <li><code>n_samples</code> ≥ <code>n_features</code></li>
  <li>-10.0 ≤ values in <code>X</code> ≤ 10.0</li>
  <li><code>y</code> contains only binary values: 0 or 1</li>
  <li>Solutions are tested with absolute tolerance of 1e-2 and relative tolerance of 1e-2</li>

  <li>Performance is measured with <code>n_features</code> = 8, <code>n_samples</code> = 16</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// X, y, beta are device pointers
extern "C" void solve(const float* X, const float* y, float* beta, int n_samples, int n_features) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# X, y, beta are tensors on the GPU
@cute.jit
def solve(
    X: cute.Tensor, y: cute.Tensor, beta: cute.Tensor, n_samples: cute.Int32, n_features: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# X, y are tensors on the GPU
@jax.jit
def solve(X: jax.Array, y: jax.Array, n_samples: int, n_features: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# X, y, beta are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    X: UnsafePointer[Float32, MutExternalOrigin],
    y: UnsafePointer[Float32, MutExternalOrigin],
    beta: UnsafePointer[Float32, MutExternalOrigin],
    n_samples: Int32,
    n_features: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# X, y, beta are tensors on the GPU
def solve(X: torch.Tensor, y: torch.Tensor, beta: torch.Tensor, n_samples: int, n_features: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# X, y, beta are tensors on the GPU
def solve(X: torch.Tensor, y: torch.Tensor, beta: torch.Tensor, n_samples: int, n_features: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Logistic Regression", atol=1e-02, rtol=1e-02, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self, X: torch.Tensor, y: torch.Tensor, beta: torch.Tensor, n_samples: int, n_features: int
    ):
        """
        Logistic regression using Newton-Raphson (IRLS) in PyTorch.
        This converges faster and more accurately than plain gradient descent.
        """
        assert X.dtype == torch.float32
        assert y.dtype == torch.float32
        assert beta.dtype == torch.float32
        assert X.shape == (n_samples, n_features)
        assert y.shape == (n_samples,)
        assert beta.shape == (n_features,)

        X_reshaped = X.view(n_samples, n_features)
        y_reshaped = y.view(n_samples)
        beta.zero_()

        max_iter = 1000
        tol = 1e-8
        l2_reg = 1e-6

        for _ in range(max_iter):
            z = torch.mv(X_reshaped, beta)
            p = torch.sigmoid(z)
            W = p * (1 - p)
            W = torch.clamp(W, min=1e-8)

            # Gradient
            gradient = torch.mv(X_reshaped.t(), p - y_reshaped) + l2_reg * beta

            # Hessian
            XW = X_reshaped * W.unsqueeze(1)
            hessian = torch.mm(X_reshaped.t(), XW) + l2_reg * torch.eye(
                n_features, device=X.device, dtype=X.dtype
            )

            # Solve H @ delta = gradient
            try:
                delta = torch.linalg.solve(hessian, gradient)
            except RuntimeError:
                delta = torch.linalg.lstsq(hessian, gradient.unsqueeze(1)).solution.squeeze()

            beta_new = beta - delta

            if torch.norm(beta_new - beta) < tol:
                beta.copy_(beta_new)
                break

            beta.copy_(beta_new)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "X": (ctypes.POINTER(ctypes.c_float), "in"),
            "y": (ctypes.POINTER(ctypes.c_float), "in"),
            "beta": (ctypes.POINTER(ctypes.c_float), "out"),
            "n_samples": (ctypes.c_int, "in"),
            "n_features": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        X = torch.tensor(
            [
                [2.0, 1.0],
                [1.0, 2.0],
                [3.0, 3.0],
                [1.5, 2.5],
                [-1.0, -2.0],
                [-2.0, -1.0],
                [-1.5, -2.5],
                [-3.0, -3.0],
            ],
            device="cuda",
            dtype=dtype,
        )
        y = torch.tensor([1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0], device="cuda", dtype=dtype)
        beta = torch.zeros(2, device="cuda", dtype=dtype)
        return {
            "X": X,
            "y": y,
            "beta": beta,
            "n_samples": 8,
            "n_features": 2,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # simple_1d
        tests.append(
            {
                "X": torch.tensor(
                    [
                        [0.24799999594688416],
                        [-0.0689999982714653],
                        [0.3240000009536743],
                        [0.7620000243186951],
                        [-0.11699999868869781],
                    ],
                    device="cuda",
                    dtype=dtype,
                ),
                "y": torch.tensor([1.0, 1.0, 0.0, 0.0, 0.0], device="cuda", dtype=dtype),
                "beta": torch.zeros(1, device="cuda", dtype=dtype),
                "n_samples": 5,
                "n_features": 1,
            }
        )

        # simple_2d
        tests.append(
            {
                "X": torch.tensor(
                    [
                        [0.1289999932050705, -0.45399999618530273],
                        [-0.1889999955892563, -0.2669999897480011],
                        [0.42899999022483826, -0.2070000022649765],
                        [0.24899999797344208, 1.0049999952316284],
                        [0.6309999823570251, -0.2199999988079071],
                        [-0.17299999296665192, 0.2280000001192093],
                    ],
                    device="cuda",
                    dtype=dtype,
                ),
                "y": torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0, 0.0], device="cuda", dtype=dtype),
                "beta": torch.zeros(2, device="cuda", dtype=dtype),
                "n_samples": 6,
                "n_features": 2,
            }
        )

        # square_3x3
        tests.append(
            {
                "X": torch.tensor(
                    [
                        [0.125, 0.6579999923706055, 0.6230000257492065],
                        [-0.8019999861717224, -0.23399999737739563, -0.8579999804496765],
                        [0.9290000200271606, 0.04399999976158142, 0.4740000069141388],
                    ],
                    device="cuda",
                    dtype=dtype,
                ),
                "y": torch.tensor([1.0, 0.0, 1.0], device="cuda", dtype=dtype),
                "beta": torch.zeros(3, device="cuda", dtype=dtype),
                "n_samples": 3,
                "n_features": 3,
            }
        )

        # overdetermined_8x3
        tests.append(
            {
                "X": torch.tensor(
                    [
                        [0.013000000268220901, 0.12999999523162842, -0.1979999989271164],
                        [-0.10199999809265137, -0.6359999775886536, -1.2979999780654907],
                        [0.14499999582767487, -0.43700000643730164, 0.19699999690055847],
                        [0.46799999475479126, -0.00800000037997961, 0.12999999523162842],
                        [-0.7369999885559082, 0.4009999930858612, -0.875],
                        [-0.24799999594688416, -0.5040000081062317, 0.013000000268220901],
                        [-0.061000000685453415, -0.7730000019073486, -0.30300000309944153],
                        [-0.6970000267028809, -0.3140000104904175, 0.16599999368190765],
                    ],
                    device="cuda",
                    dtype=dtype,
                ),
                "y": torch.tensor(
                    [1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0], device="cuda", dtype=dtype
                ),
                "beta": torch.zeros(3, device="cuda", dtype=dtype),
                "n_samples": 8,
                "n_features": 3,
            }
        )

        # medium_10x3
        tests.append(
            {
                "X": torch.tensor(
                    [
                        [0.2919999957084656, 0.6159999966621399, 0.41100001335144043],
                        [-0.4000000059604645, 0.20600000023841858, -0.08799999952316284],
                        [-0.03700000047683716, -0.28299999237060547, -0.04699999839067459],
                        [0.42899999022483826, -0.4309999942779541, 0.00800000037997961],
                        [0.7829999923706055, -0.23499999940395355, -0.19599999487400055],
                        [0.40799999237060547, 0.03799999877810478, -0.05000000074505806],
                        [0.8119999766349792, -0.6679999828338623, -0.06800000369548798],
                        [-0.23899999260902405, -0.796999990940094, -0.4339999854564667],
                        [-0.01600000075995922, -0.7639999985694885, -0.06199999898672104],
                        [-0.13099999725818634, 0.49799999594688416, 0.1589999943971634],
                    ],
                    device="cuda",
                    dtype=dtype,
                ),
                "y": torch.tensor(
                    [1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0], device="cuda", dtype=dtype
                ),
                "beta": torch.zeros(3, device="cuda", dtype=dtype),
                "n_samples": 10,
                "n_features": 3,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        device = "cuda"

        X = torch.eye(8, device=device, dtype=dtype).repeat(2, 1)
        y = torch.tensor(
            [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0],
            device=device,
            dtype=dtype,
        )
        beta = torch.zeros(8, device=device, dtype=dtype)

        return {
            "X": X,
            "y": y,
            "beta": beta,
            "n_samples": 16,
            "n_features": 8,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
